In [ ]:
# Importe und Pfade
from pathlib import Path
import json
import pandas as pd
import numpy as np
import pm4py
PROJECT_ROOT = Path('..').resolve()
DATA_PATH = PROJECT_ROOT / 'data_raw' / 'BPI_Challenge_2018.xes.gz'
OUT_DIR = PROJECT_ROOT / 'outputs' / 'deviation_label_audit'
TABLE_DIR = OUT_DIR / 'tables'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
print('Project root:', PROJECT_ROOT)
print('Data path:', DATA_PATH)
print('Data exists:', DATA_PATH.exists())
print('Output:', OUT_DIR)


In [ ]:
# Log laden
log = pm4py.read_xes(str(DATA_PATH))
df = pm4py.convert_to_dataframe(log)
CASE_COL = 'case:concept:name'
ACTIVITY_COL = 'concept:name'
TIME_COL = 'time:timestamp'
df[TIME_COL] = pd.to_datetime(df[TIME_COL], errors='coerce')
df['__row_order__'] = np.arange(len(df))
if 'activity' not in df.columns:
    df['activity'] = df[ACTIVITY_COL]
df['combined_activity'] = df['doctype'].astype(str) + ' | ' + df['subprocess'].astype(str) + ' | ' + df['activity'].astype(str)
print('Shape:', df.shape)
print('Cases:', df[CASE_COL].nunique())
print('Events:', len(df))
print('Combined activities:', df['combined_activity'].nunique())
df.head()


In [ ]:
# Fallattribute
case_attrs = [c for c in df.columns if c.startswith('case:')]
case_attr_df = df.sort_values([CASE_COL, TIME_COL, '__row_order__']).groupby(CASE_COL)[case_attrs].first()
case_basic = df.groupby(CASE_COL).agg(event_count=(ACTIVITY_COL, 'size'), case_start=(TIME_COL, 'min'), case_end=(TIME_COL, 'max'), n_raw_activities=(ACTIVITY_COL, 'nunique'), n_combined_activities=('combined_activity', 'nunique'), n_docs=('docid', 'nunique'), n_doctypes=('doctype', 'nunique'), n_subprocesses=('subprocess', 'nunique'), n_resources=('org:resource', 'nunique'))
case_basic['duration_days'] = (case_basic['case_end'] - case_basic['case_start']).dt.total_seconds() / (3600 * 24)
case_basic['start_calendar_year'] = case_basic['case_start'].dt.year
case_basic['end_calendar_year'] = case_basic['case_end'].dt.year
case_df = case_basic.join(case_attr_df, how='left')
print(case_df.shape)
case_df.head()


In [ ]:
# Aktivitätshäufigkeiten
activity_counts = pd.crosstab(df[CASE_COL], df['activity'])
activity_counts.columns = [f"act__{str(c).replace(' ', '_').replace('-', '_')}" for c in activity_counts.columns]
subprocess_counts = pd.crosstab(df[CASE_COL], df['subprocess'])
subprocess_counts.columns = [f"subprocess__{str(c).replace(' ', '_').replace('-', '_')}" for c in subprocess_counts.columns]
doctype_counts = pd.crosstab(df[CASE_COL], df['doctype'])
doctype_counts.columns = [f"doctype__{str(c).replace(' ', '_').replace('-', '_')}" for c in doctype_counts.columns]
combined_counts = pd.crosstab(df[CASE_COL], df['combined_activity'])
combined_counts.columns = ['comb__' + str(c).replace(' | ', '__').replace(' ', '_').replace('-', '_').replace('/', '_') for c in combined_counts.columns]
case_df = case_df.join(activity_counts, how='left').join(subprocess_counts, how='left').join(doctype_counts, how='left')
combined_counts.to_csv(TABLE_DIR / 'combined_activity_counts_by_case.csv', encoding='utf-8-sig')
count_cols = [c for c in case_df.columns if c.startswith(('act__', 'subprocess__', 'doctype__'))]
case_df[count_cols] = case_df[count_cols].fillna(0).astype(int)
print('Case df with counts:', case_df.shape)
case_df.head()


In [ ]:
# Wiederholungen
per_case_act = df.groupby([CASE_COL, 'activity']).size().reset_index(name='count')
rep_act = per_case_act[per_case_act['count'] > 1].copy()
rep_summary = rep_act.groupby(CASE_COL).agg(repeated_activity_types=('activity', 'nunique'), repeated_activity_total_extra=('count', lambda x: int((x - 1).sum())), max_repeat_single_activity=('count', 'max'))
per_case_comb = df.groupby([CASE_COL, 'combined_activity']).size().reset_index(name='count')
rep_comb = per_case_comb[per_case_comb['count'] > 1].copy()
rep_comb_summary = rep_comb.groupby(CASE_COL).agg(repeated_combined_activity_types=('combined_activity', 'nunique'), repeated_combined_activity_total_extra=('count', lambda x: int((x - 1).sum())), max_repeat_single_combined_activity=('count', 'max'))
case_df = case_df.join(rep_summary, how='left').join(rep_comb_summary, how='left')
rework_cols = ['repeated_activity_types', 'repeated_activity_total_extra', 'max_repeat_single_activity', 'repeated_combined_activity_types', 'repeated_combined_activity_total_extra', 'max_repeat_single_combined_activity']
case_df[rework_cols] = case_df[rework_cols].fillna(0)
case_df[rework_cols + ['event_count', 'duration_days']].describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]).T


In [ ]:
# Fehlende Aktivitätsspalten ergänzen
def has_col(name):
    return name in case_df.columns

def col(name):
    if name in case_df.columns:
        return case_df[name]
    return pd.Series(0, index=case_df.index)
duration_p90 = case_df['duration_days'].quantile(0.9)
duration_p95 = case_df['duration_days'].quantile(0.95)
event_count_p90 = case_df['event_count'].quantile(0.9)
event_count_p95 = case_df['event_count'].quantile(0.95)
combined_rework_p90 = case_df['repeated_combined_activity_total_extra'].quantile(0.9)
combined_rework_p95 = case_df['repeated_combined_activity_total_extra'].quantile(0.95)
label_df = case_df.copy()
label_df['label_temporal_duration_p90'] = label_df['duration_days'] >= duration_p90
label_df['label_temporal_duration_p95'] = label_df['duration_days'] >= duration_p95
label_df['label_structural_many_events_p90'] = label_df['event_count'] >= event_count_p90
label_df['label_structural_many_events_p95'] = label_df['event_count'] >= event_count_p95
label_df['label_structural_high_combined_rework_p90'] = label_df['repeated_combined_activity_total_extra'] >= combined_rework_p90
label_df['label_structural_high_combined_rework_p95'] = label_df['repeated_combined_activity_total_extra'] >= combined_rework_p95
label_df['label_path_change'] = col('subprocess__Change') > 0
label_df['label_path_objection'] = col('subprocess__Objection') > 0
label_df['label_path_change_or_objection'] = label_df['label_path_change'] | label_df['label_path_objection']
label_df['label_payment_abort'] = col('act__abort_payment') > 0
label_df['label_revoke_decision'] = col('act__revoke_decision') > 0
label_df['label_refuse'] = col('act__refuse') > 0
label_df['label_withdraw'] = col('act__withdraw') > 0
label_df['label_begin_editing_from_refused'] = col('act__begin_editing_from_refused') > 0
label_df['label_remove_document'] = col('act__remove_document') > 0
label_df['label_restart_editing'] = col('act__restart_editing') > 0
label_df['label_structural_exception_activity'] = label_df['label_payment_abort'] | label_df['label_revoke_decision'] | label_df['label_refuse'] | label_df['label_withdraw'] | label_df['label_begin_editing_from_refused'] | label_df['label_remove_document'] | label_df['label_restart_editing']
if 'case:rejected' in label_df.columns:
    label_df['label_case_rejected'] = label_df['case:rejected'].astype(bool)
else:
    label_df['label_case_rejected'] = False
label_df['label_composite_temporal_or_structural_p90'] = label_df['label_temporal_duration_p90'] | label_df['label_structural_many_events_p90'] | label_df['label_structural_high_combined_rework_p90'] | label_df['label_path_change_or_objection'] | label_df['label_structural_exception_activity']
label_df['label_composite_strict_p95'] = label_df['label_temporal_duration_p95'] | label_df['label_structural_many_events_p95'] | label_df['label_structural_high_combined_rework_p95'] | label_df['label_path_objection'] | label_df['label_revoke_decision'] | label_df['label_refuse']
thresholds = {'duration_p90': float(duration_p90), 'duration_p95': float(duration_p95), 'event_count_p90': float(event_count_p90), 'event_count_p95': float(event_count_p95), 'combined_rework_p90': float(combined_rework_p90), 'combined_rework_p95': float(combined_rework_p95)}
thresholds


In [ ]:
# Labelprävalenz
label_cols = [c for c in label_df.columns if c.startswith('label_')]
label_prevalence = pd.DataFrame({'label': label_cols, 'cases_true': [int(label_df[c].sum()) for c in label_cols], 'share_pct': [float(label_df[c].mean() * 100) for c in label_cols]}).sort_values('share_pct', ascending=False)
label_prevalence.to_csv(TABLE_DIR / 'label_prevalence.csv', index=False, encoding='utf-8-sig')
label_prevalence


In [ ]:
# Labelüberschneidungen
overlap = pd.DataFrame(index=label_cols, columns=label_cols, dtype=float)
for a in label_cols:
    for b in label_cols:
        overlap.loc[a, b] = (label_df[a] & label_df[b]).mean() * 100
overlap.to_csv(TABLE_DIR / 'label_overlap_pct_all_cases.csv', encoding='utf-8-sig')
overlap.round(2)


In [ ]:
# Gruppenvergleiche
group_cols = []
if 'case:year' in label_df.columns:
    group_cols.append('case:year')
if 'case:department' in label_df.columns:
    group_cols.append('case:department')
if group_cols:
    rows = []
    for keys, grp in label_df.groupby(group_cols, dropna=False):
        if not isinstance(keys, tuple):
            keys = (keys,)
        base = dict(zip(group_cols, keys))
        base['n_cases'] = len(grp)
        for c in label_cols:
            base[c + '_share_pct'] = float(grp[c].mean() * 100)
        rows.append(base)
    label_by_group = pd.DataFrame(rows)
    label_by_group.to_csv(TABLE_DIR / 'label_by_year_department.csv', index=False, encoding='utf-8-sig')
    display(label_by_group.head(20))
else:
    label_by_group = pd.DataFrame()
    print('No year/department columns found.')


In [ ]:
# Falldaten speichern
case_level_path = TABLE_DIR / 'case_level_features_with_label_candidates.csv'
label_df.to_csv(case_level_path, encoding='utf-8-sig')
with open(TABLE_DIR / 'label_thresholds.json', 'w', encoding='utf-8') as f:
    json.dump(thresholds, f, indent=2, ensure_ascii=False)
print('Saved:', case_level_path)
print('Rows:', len(label_df), 'Columns:', label_df.shape[1])
